# 📘 부록: Hugging Face 허브에 노무 상담 어댑터와 데이터셋 올리기

교안 01에서 저장한 노무 상담 LoRA 어댑터와 학습 데이터를 Hugging Face 허브의 내 저장소에 공개로 올리고, 저장소 이름으로 다시 받아 씁니다. 파드를 종료해도 허브에 올린 결과는 남아 다른 파드나 팀원이 이어서 쓸 수 있습니다. 그림의 빨간 상자가 설명에서 가리키는 곳입니다.

**실행 환경:** 교안 01을 끝까지 실행한 파드에서 실행합니다.

## 1. 허브와 저장소

- **Hugging Face 허브(Hub):** 모델과 데이터셋을 저장소 단위로 올리고 받는 사이트(huggingface.co)입니다. `from_pretrained("Qwen/Qwen3.5-0.8B")`가 모델을 받아 온 곳입니다.
- **저장소(repository):** 이름(repo id)은 `계정/저장소 이름`이고, 종류는 model과 dataset을 씁니다. 올릴 때마다 커밋(버전)이 하나 남습니다.
- **공개(public)와 비공개(private):** 공개 저장소는 누구나 보고 받을 수 있고, 비공개 저장소는 내 토큰으로만 보입니다. 이 부록은 공개로 올리고(공개한 법령 조문으로 만든 Q&A), 회사 데이터처럼 밖에 내면 안 되는 것은 올리는 함수에 `private=True`를 넘겨 비공개로 올립니다.

| 올리는 것 | 로컬 위치 | 저장소 종류 | 저장소 이름 |
|---|---|---|---|
| 노무 상담 데이터(학습, 검증) | `data/본실습/labor_train.jsonl`, `labor_val.jsonl` | dataset | `<계정>/labor-counsel-qa` |
| 노무 상담 LoRA 어댑터 | `output/labor_lora_adapter` | model | `<계정>/qwen3.5-0.8b-labor-lora` |

## 2. 쓰기 토큰 만들기

올리려면 쓰기(Write) 권한이 있는 **접근 토큰(access token)** 이 필요합니다. 설정 화면은 처음 열 때 비밀번호를 한 번 더 묻습니다.

- 누를 곳: 오른쪽 위 프로필 사진 > Settings > 왼쪽 메뉴 Access Tokens > Create new token > Write
- 주소: `https://huggingface.co/settings/tokens/new?tokenType=write`

<img src="images/hf/hf_01_token_write.png" width="900" alt="토큰 만들기 화면: Write 탭과 Create token 버튼에 빨간 상자">

- **Token type:** `Write`를 고릅니다. `Read` 토큰으로 올리면 403 오류가 납니다.
- **Create token:** `hf_`로 시작하는 값은 한 번만 보이니 바로 복사해 단원 폴더 `.env`에 `HF_TOKEN=복사한_토큰`으로 적습니다(`.env`는 git에 올라가지 않습니다).

## 3. 실습 준비

`HfApi`는 저장소 만들기, 올리기, 조회 같은 허브 작업을 부르는 클라이언트입니다. `whoami()`로 토큰의 계정과 권한(write)을 확인하고 저장소 이름을 정합니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import HfApi

material_dir = Path(".")  # data가 있는 단원 폴더
adapter_dir = material_dir / "output" / "labor_lora_adapter"  # 교안 01 6-2에서 save_pretrained 로 저장한 어댑터

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")
hf_token = os.getenv("HF_TOKEN")

# 토큰의 계정과 권한을 봅니다. 올리려면 권한이 write 여야 합니다
api = HfApi(token=hf_token)
account = api.whoami()
user_name = account["name"]
print("계정:", user_name, "| 권한:", account["auth"]["accessToken"]["role"])

# 계정 이름 뒤에 저장소 이름을 붙입니다(베이스 모델과 도메인이 이름에서 보이게)
dataset_repo_id = f"{user_name}/labor-counsel-qa"
model_repo_id = f"{user_name}/qwen3.5-0.8b-labor-lora"
print(dataset_repo_id, "|", model_repo_id)

## 4. 데이터셋 올리기

### 4-1. push_to_hub

`load_dataset("json", data_files=...)`는 split(train, validation)마다 `Dataset` 하나를 담은 `DatasetDict`를 만듭니다. `push_to_hub(repo_id)`는 저장소가 없으면 만들고 split마다 Parquet 파일로 바꿔 한 커밋으로 올립니다.

In [ ]:
from datasets import load_dataset

# 교안 01이 쓴 학습, 검증 파일을 split 이름과 함께 읽습니다
labor_dataset = load_dataset("json", data_files={
    "train": str(material_dir / "data" / "본실습" / "labor_train.jsonl"),
    "validation": str(material_dir / "data" / "본실습" / "labor_val.jsonl"),
})
display(labor_dataset)

# 공개 저장소를 만들고 두 split을 올립니다("Setting num_proc ..." 빨간 줄은 오류가 아닌 안내입니다)
data_commit = labor_dataset.push_to_hub(dataset_repo_id, private=False, token=hf_token,  # private=True 면 비공개 저장소
                                        commit_message="노무 상담 Q&A 학습, 검증 데이터")
print("커밋 주소:", data_commit.commit_url)

### 4-2. 데이터셋 카드

**데이터셋 카드(dataset card)** 는 저장소의 `README.md`이고 허브 첫 화면에 보이는 설명서입니다. 위쪽 메타데이터는 허브 검색과 필터에 쓰이고, 아래 본문은 데이터를 받아 쓸 사람이 읽습니다. `push_to_hub`가 메타데이터에 split 정보만 적었으므로, 나머지를 채워 다시 올립니다.

**카드에 넣는 것**(파인튜닝 데이터셋 단원의 데이터 카드와 같은 항목)
- **메타데이터:** 언어(`language`), 작업 종류(`task_categories`), 알면 라이선스(`license`)
- **목적:** 무엇을 학습하려고 만든 데이터인지
- **출처와 만든 방법:** 원본 자료, 생성·증강·필터 같은 처리
- **열과 규모:** 학습에 쓰는 열, split별 행 수
- **한계:** 남아 있을 수 있는 오류, 쓰면 안 되는 곳

In [ ]:
from huggingface_hub import DatasetCard

# push_to_hub 가 만든 README.md 를 불러와 메타데이터와 본문을 채웁니다
dataset_card = DatasetCard.load(dataset_repo_id, token=hf_token)
dataset_card.data.language = ["ko"]
dataset_card.data.task_categories = ["text-generation"]

train_count = len(labor_dataset["train"])
val_count = len(labor_dataset["validation"])
dataset_card.text = f"""
# 노무 상담 Q&A

노동법 조문에 근거해 답하는 노무 상담 챗봇의 지도 파인튜닝(SFT) 데이터입니다.

- 출처: 국가법령정보센터(law.go.kr) 노동법 조문으로 LLM이 만든 Q&A와 기존 FAQ, 그 증강 행
- 학습에 쓰는 열: `messages`(system, user, assistant)
- 규모: train {train_count:,}행, validation {val_count:,}행
- 한계: LLM이 만든 질문과 답이 섞여 사실 오류가 있을 수 있어 법률 자문으로 쓰지 않습니다.
"""
card_commit = dataset_card.push_to_hub(dataset_repo_id, token=hf_token, commit_message="데이터셋 카드 작성")
print("커밋 주소:", card_commit.commit_url)

## 5. 어댑터 올리기

### 5-1. 모델 카드

**모델 카드(model card)** 는 모델 저장소의 `README.md`입니다. `save_pretrained`가 만든 틀에 이미 `base_model`이 들어 있어 허브가 베이스 모델과의 관계를 보여 줍니다. 여기에 학습 데이터 저장소(`datasets`)와 라이선스를 더하고 본문을 씁니다.

In [ ]:
from huggingface_hub import ModelCard

# save_pretrained 가 만든 카드 틀을 불러와 메타데이터와 본문을 채우고 같은 파일에 저장합니다
model_card = ModelCard.load(adapter_dir / "README.md")
model_card.data.datasets = [dataset_repo_id]  # 허브 화면에 데이터셋 링크가 생깁니다
model_card.data.language = ["ko"]
model_card.data.license = "apache-2.0"  # 베이스 모델과 같은 라이선스
model_card.text = f"""
# 노무 상담 LoRA 어댑터

`Qwen/Qwen3.5-0.8B`에 노무 상담 Q&A(`{dataset_repo_id}`)로 학습한 LoRA 어댑터입니다(r 8, lora_alpha 16, all-linear, 학습률 2e-4, 4에폭).

쓰는 법: `PeftModel.from_pretrained(베이스 모델, "{model_repo_id}")`
"""
model_card.save(adapter_dir / "README.md")
print(model_card.data.to_yaml())

### 5-2. create_repo와 upload_folder

폴더는 `create_repo`로 저장소를 만들고(`exist_ok=True`면 이미 있어도 그대로 둡니다), `upload_folder`로 폴더 전체를 한 커밋에 올립니다. 학습을 막 끝낸 노트북이라면 `lora_model.push_to_hub(repo_id)` 한 줄로도 됩니다.

In [ ]:
# 공개 모델 저장소를 만들고 어댑터 폴더를 올립니다
api.create_repo(model_repo_id, private=False, exist_ok=True)  # private=True 면 비공개 저장소
api.upload_folder(repo_id=model_repo_id, folder_path=adapter_dir, commit_message="노무 상담 LoRA 어댑터")

# 저장소의 파일 목록
display(api.list_repo_files(model_repo_id))

## 6. 허브 화면 확인

### 6-1. 데이터셋 저장소

- 누를 곳: 오른쪽 위 프로필 사진 > 내 계정 이름 > Datasets 목록의 labor-counsel-qa
- 주소: `https://huggingface.co/datasets/<계정>/labor-counsel-qa`

<img src="images/hf/hf_02_dataset_page.png" width="900" alt="데이터셋 저장소 화면: 저장소 이름, Dataset Viewer의 split, 이 데이터로 학습한 모델에 빨간 상자">

- **저장소 이름:** 3-3에서 정한 repo id입니다. 공개 저장소라 로그인하지 않아도 이 화면이 열립니다.
- **Dataset Viewer:** 올린 Parquet 파일을 표로 보여 줍니다. split 선택 칸에서 train, validation과 행 수를 확인합니다(올린 직후에는 표를 만드는 동안 몇 분 걸립니다). 비공개 데이터셋은 무료 계정에서 이 표가 나오지 않습니다.
- **Tasks, Languages:** 4-2에서 카드에 넣은 메타데이터입니다.
- **Models trained or fine-tuned on:** 5-1에서 모델 카드의 `datasets`에 이 저장소를 적어 연결됐습니다.

- 누를 곳: 데이터셋 화면 위쪽 Files and versions 탭
- 주소: `https://huggingface.co/datasets/<계정>/labor-counsel-qa/tree/main`

<img src="images/hf/hf_03_dataset_files.png" width="900" alt="데이터셋 파일 화면: data 폴더, .gitattributes, README.md 와 커밋 3개에 빨간 상자">

- **data 폴더와 History:** data 폴더에 split마다 Parquet 파일이 하나씩 있고, 커밋은 저장소 생성(`initial commit`), 데이터, 카드 셋입니다.

### 6-2. 모델 저장소

- 누를 곳: 오른쪽 위 프로필 사진 > 내 계정 이름 > Models 목록의 qwen3.5-0.8b-labor-lora
- 주소: `https://huggingface.co/<계정>/qwen3.5-0.8b-labor-lora`

<img src="images/hf/hf_04_model_page.png" width="900" alt="모델 저장소 화면: 저장소 이름, Model tree, 학습 데이터셋 링크에 빨간 상자">

- **Model tree:** 카드의 `base_model`을 읽어 `Qwen/Qwen3.5-0.8B`의 Adapter라고 보여 줍니다. 맨 위 `Qwen/Qwen3.5-0.8B-Base`는 Qwen의 사전학습 모델이고 이 단원은 쓰지 않습니다.
- **Dataset used to train:** 카드의 `datasets`가 만든 데이터셋 링크입니다.

- 누를 곳: 모델 화면 위쪽 Files and versions 탭
- 주소: `https://huggingface.co/<계정>/qwen3.5-0.8b-labor-lora/tree/main`

<img src="images/hf/hf_05_model_files.png" width="900" alt="모델 파일 화면: 어댑터 파일 두 개와 커밋 2개에 빨간 상자">

- **adapter_config.json, adapter_model.safetensors:** 교안 01 6-2에서 저장한 어댑터 파일 두 개입니다. 베이스 모델 없이 어댑터만 올려 약 21.7 MB입니다.

## 7. 허브에서 받아 쓰기

다른 파드나 다른 사람도 폴더 대신 repo id로 받습니다. 공개 저장소라 토큰 없이 받고, 비공개 저장소라면 `token=hf_token`을 함께 넘깁니다.

In [ ]:
# 허브에서 데이터셋을 받아 split별 행 수를 봅니다
hub_dataset = load_dataset(dataset_repo_id)
print("train:", len(hub_dataset["train"]), "| validation:", len(hub_dataset["validation"]))

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 계산 장치: GPU(cuda)가 있으면 GPU, 없으면 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# 베이스 모델을 받고 허브의 어댑터를 얹습니다(fast path 안내는 교안 01과 같은 경고입니다)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-0.8B", dtype=torch.float32).to(device)
hub_model = PeftModel.from_pretrained(base_model, model_repo_id)

# 검증 데이터 한 행의 system, user 메시지로 묻습니다(교안 01 generate_reply 와 같은 단계)
messages = hub_dataset["validation"][39]["messages"][:-1]  # 마지막 assistant(정답 답변)는 뺍니다
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    output_ids = hub_model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                    eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
                                    pad_token_id=tokenizer.pad_token_id)

# 새로 만든 토큰만 글자로 바꿉니다
print("질문:", messages[-1]["content"])
print("답:", tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 8. 필요할 때만: 비공개 저장소, 병합 모델

- **비공개 저장소:** 개인정보나 회사 문서가 든 데이터와 그것으로 학습한 어댑터는 처음부터 `private=True`로 올립니다. 이미 올린 저장소는 `api.update_repo_settings(repo_id, private=True)`(데이터셋은 `repo_type="dataset"`을 더합니다)나 저장소 Settings 탭에서 바꾸고, 받을 때는 `token=hf_token`을 넘깁니다.
- **병합 모델 올리기:** 교안 02의 병합 모델도 `upload_folder`로 올리지만 float32 모델 전체라 약 3GB입니다. 어댑터만 올리면 받는 쪽이 베이스 모델을 허브에서 따로 받습니다.